In [1]:
import pandas as pd
from Bio import AlignIO

In [2]:

# 1-letter → 3-letter conversion
aa1to3 = {
    'A': 'ALA', 'R': 'ARG', 'N': 'ASN', 'D': 'ASP',
    'C': 'CYS', 'Q': 'GLN', 'E': 'GLU', 'G': 'GLY',
    'H': 'HIS', 'I': 'ILE', 'L': 'LEU', 'K': 'LYS',
    'M': 'MET', 'F': 'PHE', 'P': 'PRO', 'S': 'SER',
    'T': 'THR', 'W': 'TRP', 'Y': 'TYR', 'V': 'VAL',
    '-': '-'  # gaps
}

In [3]:

# --- Load MSA ---
msa_file = "cluster_0_1_mixed_MSTA_aa_plus_ec6098ref_2.faa"
msa = AlignIO.read(msa_file, "fasta")
ref_seq_id = "EC6098_reference"  # replace with your reference ID


In [4]:
# --- Build reference position -> MSA column mapping ---
def build_ref_map(msa, ref_seq_id):
    ref_record = next(r for r in msa if r.id == ref_seq_id)
    ref_map = {}
    ref_pos = 1  # 1-based
    for col_idx, aa in enumerate(ref_record.seq):
        if aa != '-':
            ref_map[ref_pos] = col_idx
            ref_pos += 1
    return ref_map

ref_map = build_ref_map(msa, ref_seq_id)

# # --- Map reference positions to a target sequence ---
# def map_positions(msa, ref_map, target_seq_id):
#     target_record = next(r for r in msa if r.id == target_seq_id)
#     mapping = {}
#     for ref_pos, col_idx in ref_map.items():
#         mapping[ref_pos] = target_record.seq[col_idx]
#     return mapping


In [ ]:

# --- Expand interaction dataframe ---

df_interactions = pd.read_csv("all_contacts_with_feat_imp.tsv", sep='\t')
# df_interactions = df_interactions.loc[~df_interactions['chain_A'].str.contains('unknown')]


expanded_rows = []

for record in msa:
    if record.id == ref_seq_id:
        continue  # skip reference
    target_seq = record.seq

    # Create a mapping: MSA column -> target protein residue number
    msa_to_prot_pos = {}
    prot_pos = 1
    for col_idx, aa in enumerate(target_seq):
        if aa != '-':
            msa_to_prot_pos[col_idx] = prot_pos
            prot_pos += 1
        else:
            msa_to_prot_pos[col_idx] = None  # gap

    # Map reference positions to MSA columns
    for _, row in df_interactions.iterrows():
        new_row = row.copy()

        try:
            A_ref_pos = int(float(row.A_pos))
            B_ref_pos = int(float(row.B_pos))
        except ValueError:
            A_ref_pos = None
            B_ref_pos = None

        # Get corresponding MSA columns in reference
        A_msa_col = ref_map.get(A_ref_pos, None)
        B_msa_col = ref_map.get(B_ref_pos, None)

        # Map to target protein residue and number
        if A_msa_col is not None:
            A_aa = target_seq[A_msa_col]
            new_row["A_AA"] = aa1to3.get(A_aa, '-') if A_aa != '-' else '-'
            new_row["A_pos"] = msa_to_prot_pos[A_msa_col] or '-'
        else:
            new_row["A_AA"] = '-'
            new_row["A_pos"] = '-'

        if B_msa_col is not None:
            B_aa = target_seq[B_msa_col]
            new_row["B_AA"] = aa1to3.get(B_aa, '-') if B_aa != '-' else '-'
            new_row["B_pos"] = msa_to_prot_pos[B_msa_col] or '-'
        else:
            new_row["B_AA"] = '-'
            new_row["B_pos"] = '-'

        new_row["protein"] = record.id
        expanded_rows.append(new_row)

df_expanded = pd.DataFrame(expanded_rows)


In [15]:
df_interactions = pd.read_csv("all_contacts_with_feat_imp.tsv", sep="\t")
# df_interactions = df_interactions.loc[
#     ~df_interactions["chain_A"].str.contains("unknown")
# ].copy()

# convert once
df_interactions["A_ref_pos"] = pd.to_numeric(df_interactions["A_pos"], errors="coerce").astype("Int64")
df_interactions["B_ref_pos"] = pd.to_numeric(df_interactions["B_pos"], errors="coerce").astype("Int64")

df_interactions["A_msa_col"] = df_interactions["A_ref_pos"].map(ref_map)
df_interactions["B_msa_col"] = df_interactions["B_ref_pos"].map(ref_map)


In [22]:
df_interactions

,chain_A,A_AA,A_pos,A_atom,chain_B,B_AA,B_pos,B_atom,overlap,distance,...,gini_imp_A,seq_position_A,feat_rank_B,index_B,gini_imp_B,seq_position_B,A_ref_pos,B_ref_pos,A_msa_col,B_msa_col
0,#2//chain_id='A-53',ARG,495.0,CA,#2//chain_id='A-53',GLN,429.0,OE1,-0.133,3.433,...,8.862224,495.0,47.0,794_N,7.325678,429.0,495,429,786.0,670.0
1,#2//chain_id='A-53',ARG,495.0,CA,#2//chain_id='A-53',GLN,429.0,OE1,-0.133,3.433,...,8.862224,495.0,278.0,795_S,4.677206,429.0,495,429,786.0,670.0
2,#2//chain_id='A-53',ARG,495.0,CA,#2//chain_id='A-53',GLN,429.0,OE1,-0.133,3.433,...,8.862224,495.0,282.0,795_D,4.660951,429.0,495,429,786.0,670.0
3,#2//chain_id='A-53',ARG,495.0,CA,#2//chain_id='A-53',GLN,429.0,OE1,-0.133,3.433,...,8.862224,495.0,428.0,794_S,4.092880,429.0,495,429,786.0,670.0
4,#2//chain_id='A-53',ARG,495.0,O,#2//chain_id='A-53',SER,501.0,OG,-0.201,2.681,...,8.862224,495.0,NaN,NaN,NaN,NaN,495,501,786.0,792.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3240,#2//chain_id='A-53',LEU,502.0,CD1,#2//chain_id='A-52',THR,40.0,CG2,-0.214,3.974,...,3.906956,502.0,63.0,213_F,7.083645,40.0,502,40,793.0,53.0
3241,#2//chain_id='A-53',ARG,82.0,NH1,#2//chain_id='A-53',ASP,372.0,OD1,-0.083,2.743,...,3.906850,82.0,NaN,NaN,NaN,NaN,82,372,104.0,594.0
3242,#2//chain_id='A-53',ARG,82.0,O,#2//chain_id='A-53',VAL,422.0,CG1,-0.388,3.688,...,3.906850,82.0,329.0,777_T,4.410353,422.0,82,422,104.0,663.0
3243,#2//chain_id='A-53',ARG,82.0,O,#2//chain_id='A-53',VAL,422.0,CG1,-0.388,3.688,...,3.906850,82.0,345.0,775_Q,4.344452,422.0,82,422,104.0,663.0


In [5]:
def build_msa_to_prot_pos(seq):
    msa_to_pos = [None] * len(seq)
    pos = 0
    for i, aa in enumerate(seq):
        if aa != "-":
            pos += 1
            msa_to_pos[i] = pos
    return msa_to_pos

In [6]:
expanded_rows = []

A_cols = df_interactions["A_msa_col"].to_numpy()
B_cols = df_interactions["B_msa_col"].to_numpy()

for record in msa:
    # if record.id == ref_seq_id:
    #     continue
    target_seq = record.seq
    msa_to_prot_pos = build_msa_to_prot_pos(target_seq)

    for i, row in df_interactions.iterrows():
        new_row = row.copy()

        A_col = A_cols[i]
        B_col = B_cols[i]

        if pd.notna(A_col):
            A_col = int(A_col)
            aa = target_seq[A_col]
            new_row["A_AA"] = aa1to3.get(aa, "-") if aa != "-" else "-"
            new_row["A_pos"] = msa_to_prot_pos[A_col] or "-"
        else:
            new_row["A_AA"] = "-"
            new_row["A_pos"] = "-"

        if pd.notna(B_col):
            B_col = int(B_col)
            aa = target_seq[B_col]
            new_row["B_AA"] = aa1to3.get(aa, "-") if aa != "-" else "-"
            new_row["B_pos"] = msa_to_prot_pos[B_col] or "-"
        else:
            new_row["B_AA"] = "-"
            new_row["B_pos"] = "-"

        new_row["protein"] = record.id
        expanded_rows.append(new_row)


NameError: name 'df_interactions' is not defined

In [24]:
df_expanded = pd.DataFrame(expanded_rows)

In [25]:
df_expanded.loc[df_expanded['protein'].str.contains("ref")]

,chain_A,A_AA,A_pos,A_atom,chain_B,B_AA,B_pos,B_atom,overlap,distance,...,seq_position_A,feat_rank_B,index_B,gini_imp_B,seq_position_B,A_ref_pos,B_ref_pos,A_msa_col,B_msa_col,protein
0,#2//chain_id='A-53',ARG,495,CA,#2//chain_id='A-53',GLN,429,OE1,-0.133,3.433,...,495.0,47.0,794_N,7.325678,429.0,495,429,786.0,670.0,EC6098_reference
1,#2//chain_id='A-53',ARG,495,CA,#2//chain_id='A-53',GLN,429,OE1,-0.133,3.433,...,495.0,278.0,795_S,4.677206,429.0,495,429,786.0,670.0,EC6098_reference
2,#2//chain_id='A-53',ARG,495,CA,#2//chain_id='A-53',GLN,429,OE1,-0.133,3.433,...,495.0,282.0,795_D,4.660951,429.0,495,429,786.0,670.0,EC6098_reference
3,#2//chain_id='A-53',ARG,495,CA,#2//chain_id='A-53',GLN,429,OE1,-0.133,3.433,...,495.0,428.0,794_S,4.092880,429.0,495,429,786.0,670.0,EC6098_reference
4,#2//chain_id='A-53',ARG,495,O,#2//chain_id='A-53',SER,501,OG,-0.201,2.681,...,495.0,NaN,NaN,NaN,NaN,495,501,786.0,792.0,EC6098_reference
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3240,#2//chain_id='A-53',LEU,502,CD1,#2//chain_id='A-52',THR,40,CG2,-0.214,3.974,...,502.0,63.0,213_F,7.083645,40.0,502,40,793.0,53.0,EC6098_reference
3241,#2//chain_id='A-53',ARG,82,NH1,#2//chain_id='A-53',ASP,372,OD1,-0.083,2.743,...,82.0,NaN,NaN,NaN,NaN,82,372,104.0,594.0,EC6098_reference
3242,#2//chain_id='A-53',ARG,82,O,#2//chain_id='A-53',VAL,422,CG1,-0.388,3.688,...,82.0,329.0,777_T,4.410353,422.0,82,422,104.0,663.0,EC6098_reference
3243,#2//chain_id='A-53',ARG,82,O,#2//chain_id='A-53',VAL,422,CG1,-0.388,3.688,...,82.0,345.0,775_Q,4.344452,422.0,82,422,104.0,663.0,EC6098_reference


In [25]:
df_expanded.to_csv("dataframe_relativesEC6098_proteins_mapped_interactions.tsv", sep='\t', index=False)


In [26]:
df_expanded = df_expanded.loc[~((df_expanded["A_AA"] == '-') | (df_expanded["B_AA"] == '-'))]

In [27]:
data_df = pd.read_csv("/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/RF_models_automation/RF_results/raw/cluster_phyloglm_model_data_df.tsv", sep='\t',usecols=['protein','ecosystem_subtype'])
data_df

,protein,ecosystem_subtype
0,IMGVR_UViG_3300044301_001603|3300044301|Ga0466...,Oceanic
1,IMGVR_UViG_3300012919_001336|3300012919|Ga0160...,Oceanic
2,IMGVR_UViG_3300012771_000213|3300012771|Ga0138...,Lake
3,IMGVR_UViG_3300013295_000235|3300013295|Ga0170...,Lake
4,IMGVR_UViG_3300044270_001180|3300044270|Ga0466...,Oceanic
...,...,...
1246,IMGVR_UViG_3300012714_000124|3300012714|Ga0157...,Lake
1247,IMGVR_UViG_3300025428_000003|3300025428|Ga0208...,Lake
1248,IMGVR_UViG_3300003800_000017|3300003800|Ga0007...,Lake
1249,IMGVR_UViG_3300003800_000017|3300003800|Ga0007...,Lake


In [28]:
df_expand_biomes = df_expanded.set_index('protein').join(data_df.set_index('protein'),how='left')

In [29]:
df_expand_biomes

,chain_A,A_AA,A_pos,A_atom,chain_B,B_AA,B_pos,B_atom,overlap,distance,interaction,feat_rank,index,gini_imp,seq_position,side,ecosystem_subtype
protein,,,,,,,,,,,,,,,,,
IMGVR_UViG_3300044270_001180|3300044270|Ga0466304_002545_2740_4818,#2//chain_id='A-53',TRP,551,NH1,#2//chain_id='A-53',GLN,578,OE2,-0.039,2.699,self,1,800_T,5.176608,434.0,A,Oceanic
IMGVR_UViG_3300044270_001180|3300044270|Ga0466304_002545_2740_4818,#2//chain_id='A-53',TRP,551,CD,#2//chain_id='A-53',GLN,578,CD,-0.056,3.816,self,1,800_T,5.176608,434.0,A,Oceanic
IMGVR_UViG_3300044270_001180|3300044270|Ga0466304_002545_2740_4818,#2//chain_id='A-53',TRP,551,NH1,#2//chain_id='A-53',GLN,578,CD,-0.168,3.688,self,1,800_T,5.176608,434.0,A,Oceanic
IMGVR_UViG_3300044270_001180|3300044270|Ga0466304_002545_2740_4818,#2//chain_id='A-53',TRP,551,CD,#2//chain_id='A-53',GLN,578,OE1,-0.208,3.508,self,1,800_T,5.176608,434.0,A,Oceanic
IMGVR_UViG_3300044270_001180|3300044270|Ga0466304_002545_2740_4818,#2//chain_id='A-53',TRP,551,CG,#2//chain_id='A-53',ASN,604,CG,-0.253,4.013,self,1,800_T,5.176608,434.0,A,Oceanic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
IMGVR_UViG_3300047492_000094|3300047492|Ga0485327_006360_1250_2854,#2//chain_id='A-53',CYS,147,CB,#2//chain_id='A-53',VAL,111,CG2,0.262,3.498,self,837,327_P,0.353765,117.0,B,Lake
IMGVR_UViG_3300047492_000094|3300047492|Ga0485327_006360_1250_2854,#2//chain_id='A-53',GLN,113,CG,#2//chain_id='A-53',VAL,111,O,-0.279,3.579,self,837,327_P,0.353765,117.0,B,Lake
IMGVR_UViG_3300047492_000094|3300047492|Ga0485327_006360_1250_2854,#2//chain_id='A-53',PHE,96,CZ,#2//chain_id='A-53',VAL,111,CD1,-0.385,4.025,self,837,327_P,0.353765,117.0,B,Lake


In [30]:
df_expand_biomes.reset_index().to_csv("dataframe_relativesEC6098_proteins_mapped_interactions_w_biomes.tsv", sep='\t', index=False)